# Application of Machine Learning Methods to Evaluate Collectable Merchandise
**Course:** UE24CS352A — Machine Learning Mini-Project  
**Authors:** AR Akshay Kumar (PES1UG24CS705) & Shreyas Gowda (PES1UG24CS704)  

This interactive notebook demonstrates the complete end-to-end ML pipeline:
1. Data Ingestion & Exploratory Analysis
2. Feature Engineering & Preprocessing
3. Baseline Linear SVM vs. Custom Kernel SVM
4. Decision Tree & Bagged Decision Tree Ensemble
5. Live Evaluation & Predictions

In [ ]:
import sys
import os
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset, PRICE_BIN_EDGES
from src.preprocessing import CollectablesPreprocessor
from src.kernels import CustomCollectableKernel
from src.models import BaselineLinearSVM, CustomKernelSVM, VanillaDecisionTree, BaggedDecisionTree, RandomForestModel
from src.evaluation import compute_metrics, format_metrics_table

%matplotlib inline
sns.set_theme(style="whitegrid")
print("Modules imported successfully.")

## 1. Data Ingestion & Distribution

In [ ]:
df = load_dataset()
print(f"Total Collectible Items: {len(df)}")
display(df.head())
print("\nPrice Bins Distribution:")
display(df["price_bin"].value_counts().sort_index())

## 2. Model Training & Evaluation Benchmark

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(df, df["price_bin"]))

train_df = df.iloc[train_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)

preprocessor = CollectablesPreprocessor(random_state=42)
preprocessor.fit(train_df)

X_train_ohe = preprocessor.transform_one_hot(train_df)
X_test_ohe = preprocessor.transform_one_hot(test_df)
train_x_cont, train_z_cat = preprocessor.transform_ordinal(train_df)
test_x_cont, test_z_cat = preprocessor.transform_ordinal(test_df)

y_train = train_df["price_bin"].values
y_test = test_df["price_bin"].values

# 1. Linear SVM
lin_svm = BaselineLinearSVM(C=1.0, random_state=42)
lin_svm.fit(X_train_ohe, y_train)
m_lin = compute_metrics(y_test, lin_svm.predict(X_test_ohe))

# 2. Custom Kernel SVM
ck_calc = CustomCollectableKernel(marginal_probs=preprocessor.marginal_probs, weights=[2.0, 1.0, 1.0, 3.0])
ck_svm = CustomKernelSVM(kernel_calculator=ck_calc, C=1.5, random_state=42)
ck_svm.fit((train_x_cont, train_z_cat), y_train)
m_ck = compute_metrics(y_test, ck_svm.predict((test_x_cont, test_z_cat)))

# 3. Bagged Decision Tree
bag_tree = BaggedDecisionTree(n_estimators=100, random_state=42)
bag_tree.fit(X_train_ohe, y_train)
m_bag = compute_metrics(y_test, bag_tree.predict(X_test_ohe))

results_df = pd.DataFrame({
    "Linear SVM": [m_lin["weighted_precision"], m_lin["weighted_recall"], m_lin["weighted_f1"], m_lin["per_class_precision"]["Bin 4"]],
    "Custom Kernel SVM": [m_ck["weighted_precision"], m_ck["weighted_recall"], m_ck["weighted_f1"], m_ck["per_class_precision"]["Bin 4"]],
    "Bagged DT (100)": [m_bag["weighted_precision"], m_bag["weighted_recall"], m_bag["weighted_f1"], m_bag["per_class_precision"]["Bin 4"]],
}, index=["Weighted Precision", "Weighted Recall", "Weighted F1", "Bin 4 (Grail) Precision"]).T

display(results_df.round(3))